In [ ]:
#Input is Pose data from MediaPipe Pose 

In [ ]:
# 1. Normalize the Pose
# Once per clip

def normalize_pose(pose):
    #pose: (T,J,2)
    left_hip, right_hip = 23, 24 #Mediapipe Indices
    pelvis = (pose[:, left_hip]+ pose[:, right_hip]) / 2

    #Translate: Express every joint position relative to the pelvis at each moment in time. Removing the pelvis position from all joints positions per frame centers the pose data dynamically. 
    # It remove the following effects: camera panning: distance-to.camera-effects, subject walking across the frame. 
    # What remains are relative joint motion, inter-limb coordination, postural deviations.
    pose_centered = pose - pelvis[:, None, :]
    # pose.shape == (T,J,2), pelvis.shape == (T,2). The above takes all frames, adds a new placeholder axis, then takes x,y coordinates -> (T,1,2).

    #Scale by torso length per frame
    # These joints are: more stable during gait, rarely occluded, Less noisy or affected by pathology than joints used for leg length measure,

    left_shoulder, right_shoulder = 11, 12
    torso = (pose[:, left_shoulder] + pose[:, right_shoulder]) / 2
    scale = np.linalg.norm(torso - pelvis, axis=1).mean() #calculates the torso length based on the average measures across time to get stable scale per clip.

    return pose_centered / scale # All coordinates are now expressed in units of torso_length, making them more comparable across subjects

"""
#Removed:
✔ Body size differences
✔ Camera distance effects
Preserved:
✔ Relative joint motion
✔ Asymmetry
✔ Trunk lean & pelvic drop
✔ Temporal dynamics
"""


In [ ]:
#Sanity Checks:
# CHeck the hip centered trajectory oscillates around zeo
np.mean(pose_centered[:, left_hip], axis=0)
# should be close to [0, 0]

np.mean(np.linalg.norm(torso - pelvis, axis=1))
# should be ~1.0
    """
    Plot:
Knee trajectories from different subjects
They should be comparable in scale
    """

In [ ]:
# 2. Joint Selection - Research has shown that being selective about joints can improve outcomes because it reduces noise. 
# Given our limited number of output anomalies that we want to predict (due to limited input data), we focus on the following joints, most relevant for gait analysis:
# Store as joints.py ??

# A. Define joint indices

GAIT_JOINTS = [
    2, 5,     # eyes (head orientation)
    11, 12,   # shoulders
    23, 24,   # hips
    25, 26,   # knees
    27, 28,   # ankles
    29, 30,   # heels
    31, 32    # foot inde
]

JOINT_NAMES = {
    2:  "left_eye",
    5:  "right_eye",

    11: "left_shoulder",
    12: "right_shoulder",

    23: "left_hip",
    24: "right_hip",

    25: "left_knee",
    26: "right_knee",

    27: "left_ankle",
    28: "right_ankle",

    29: "left_heel",
    30: "right_heel",

    31: "left_foot_index",
    32: "right_foot_index"
}

GAIT_JOINT_GROUPS = {
    "head": {
        2: "left_eye",
        5: "right_eye"
    },

    "trunk": {
        11: "left_shoulder",
        12: "right_shoulder",
        23: "left_hip",
        24: "right_hip"
    },

    "lower_limbs": {
        25: "left_knee",
        26: "right_knee",
        27: "left_ankle",
        28: "right_ankle",
        29: "left_heel",
        30: "right_heel",
        31: "left_foot_index",
        32: "right_foot_index"
    }
}



In [ ]:
# B. Joint selection function 

def select_joints(pose, joint_indices):
    pose = np.asarray(pose)
    assert pose.ndim == 3 and pose.shape[1] >= max(joint_indices) + 1
    return pose[:, joint_indices, :]

#call function with:
reduced_pose = select_joints(pose, GAIT_JOINTS)

In [ ]:
# 3. Windowing 

#Either we normalise the data to 30 fps or we have to reference the fps for each clip here

def sliding_windows(pose, window_size=(FPS*2), stride=(FPS)):
    T = pose.shape[0]
    windows = []
    
    for start in range0, T - window_size +1, stride):

#Output:
windows.shape == (N_windows, window_size, J, 2)